# Retrieval Augmented Generation (RAG) and Vector Databases

### 1. Creating our Knowledge Base 
Load the document files and store them as Pandas DataFrame.

In [109]:
# Import Libraries
import os
import re
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict, Tuple, Optional
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

# Initialize Pinecone client
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"),
    host="http://localhost:5080",
    #host=os.getenv("PINECONE_HOST_URL")
)

# Pinecone configuration
INDEX_NAME = "markdown-document"
NAMESPACE = "docs"
DEPLOYMENT = "text-embedding-ada-002"
CHUNK_MAX_LEN = 500
CHUNK_MIN_LEN = 300
CHUNK_OVERLAP = 50

# Store data documents
data_paths = ["../data/frameworks.md", "../data/own_framework.md", "../data/perceptron.md"]

def create_dataframe(data_paths):
    # Initialize the DataFrame
    df = pd.DataFrame(columns=['path', 'text'])
    for path in data_paths:
        with open(path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        new_row = pd.DataFrame({
            'path': [path], 'text': [file_content]
        })
        df = pd.concat([df, new_row], ignore_index=True)
    return df


### 2. Chunk the text
Split the text document into overlapping chunks.

In [110]:
# Define function to split the text
def split_text(text, max_len, min_len):
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if (len(' '.join(current_chunk)) < max_len) and (len(' '.join(current_chunk)) > min_len):
            chunks.append(' '.join(current_chunk))
            current_chunk = []

    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks


### 3. Generate the Embeddings using OpenAI API
Get the embeddings for chunked text using OpenAI API.

In [111]:
# Configure DeepSeek as used LLM
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

# Define function to create embeddings
def create_embeddings(text, model=DEPLOYMENT):
    # Create embeddings for each document chunk
    embeddings = client.embeddings.create(model=model, input=text).data[0].embedding
    return embeddings


### 4. Create Pinecone Database
Create Pinecone database to store the document embeddings.

In [112]:
# Define function to create Pinecone index
def setup_pinecone_index():
    existing_indexes = pc.list_indexes().names()

    if INDEX_NAME not in existing_indexes:
        pc.create_index(
            name=INDEX_NAME,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        print(f"Created index: {INDEX_NAME}")
    else:
        print(f"Index {INDEX_NAME} already exists")

    # Get the actual host assigned by Pinecone local
    index_info = pc.describe_index(INDEX_NAME)
    host = index_info.host
    if not host.startswith("http"):
        host = f"http://{host}"
    
    print(f"Connecting to index at: {host}")
    return pc.Index(name=INDEX_NAME, host=host)


# Define function to store data documents to Pinecone
def index_documents(index, df: pd.DataFrame, metadata_columns: Optional[List[str]]=None, text_column: str="text", id_prefix: str="doc"):
    vectors_to_upsert = []
    vector_id = 0
    
    for idx, row in df.iterrows():
        # Extract the main text content
        text_content = row[text_column]

        # Determine the metadata fields
        if metadata_columns is None:
            metadata_fields = [col for col in df.columns if col != text_column]
        else:
            metadata_fields = metadata_columns

        # Build metadata dic
        base_metadata = {}
        for field in metadata_fields:
            if field in row:
                value = row[field]
                if isinstance(value, (list, dict)):
                    base_metadata[field] = str(value)
                else:
                    base_metadata[field] = value

        # Split into chunks
        chunks = split_text(text_content, CHUNK_MAX_LEN, CHUNK_MIN_LEN)

        # Generate embeddings
        for i, chunk in enumerate(chunks):
            embedding = create_embeddings(chunk, model=DEPLOYMENT)
            
            # Create unique ID
            doc_id = f"{id_prefix}_{idx}_chunk_{i}"

            # Create metadata for this chunk
            chunk_metadata = {
                **base_metadata,
                "chunk_index": i,
                "content": chunk,
                "original_row_index": idx
            }

            # Add to batch vector
            vectors_to_upsert.append((
                doc_id,
                embedding,
                chunk_metadata
            ))
            vector_id += 1
            
            if len(vectors_to_upsert) >= 100:
                index.upsert(vectors=vectors_to_upsert, namespace=NAMESPACE)
                print(f"Upserted {len(vectors_to_upsert)} vectors.")
                vectors_to_upsert = []

    # Upsert remaining vectors
    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert, namespace=NAMESPACE)
        print(f"Upserted  final {len(vectors_to_upsert)} vectors.")
    
    print(f"Total vectors indexed: {vector_id}")
    return vector_id


### 5. Retrieve the data from database
Retrieve the results from database based on user query.

In [119]:
# Define function to query Pinecone for documents
def query_pinecone(index, query: str, top_k: int=3):
    # Generate embedding for query
    query_embedding = create_embeddings(query, model=DEPLOYMENT)

    # Query Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        metric="cosine",
        namespace=NAMESPACE,
        include_metadata=True
    )

    retrieved_docs = []
    for match in results["matches"]:
        #print(match)
        retrieved_docs.append({
            "filename": match["metadata"]["path"],
            "content": match["metadata"]["content"],
            "score": match["score"]
        })
    return retrieved_docs

### 6. Putting together to answer a question
Put everything together to answer user question and provide answer based on knowledge base documents.

In [120]:
# Define the main function for chatbot
def chatbot():
    print(f"RAG System with Pinecone database and Markdown Documents")

    # Step 1: Load the Markdown files
    print(f"\nLoading Markdown files...")
    file_df = create_dataframe(data_paths)

    # Step 2: Setup Pinecone index
    print(f"\nSetting up Pinecone index...")
    index = setup_pinecone_index()  

    # Step 3: Index documents
    print(f"\nIndexing documents to Pinecone...")
    index_documents(index, file_df)

    # Step 4: Interactive Q&A loop
    print(f"\nReady for questions! (type 'exit' to exit the program)")
    print("-" * 20)

    while True:
        user_query = input("What is your question: ")
        if user_query in ["quit", "exit", "q"]:
            break
        
        print(f"Searching documentation...")
        retrieved_docs = query_pinecone(index, user_query)
    
        for i, doc in enumerate(retrieved_docs, 1):
            print(f" {i}. {doc['filename']}")
            print(f" {doc['content']}")
        
        

### 7. Run the chatbot application

In [121]:
chatbot()

RAG System with Pinecone database and Markdown Documents

Loading Markdown files...

Setting up Pinecone index...
Index markdown-document already exists
Connecting to index at: http://localhost:5081

Indexing documents to Pinecone...
Upserted  final 57 vectors.
Total vectors indexed: 57

Ready for questions! (type 'exit' to exit the program)
--------------------
Searching documentation...
 1. ../data/perceptron.md
 # Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,
 2. ../data/perceptron.md
 if z < 0: # positive example classified as negative weights = weights + eta*weights.shape z = np.dot(neg, weights) if z >= 0: # negative example classified as positive weights = weights - eta*weights.shape return weights ``` ## Conclusion In this l